# 02 - NDVI and EVI rasters from HLS

Builds cloud-masked NDVI and EVI GeoTIFFs for every HLS scene (Landsat L30 and Sentinel-2 S30),
then clips them to the study area and fills the days between overpasses by linear interpolation.
The daily rasters are the inputs for the ETa maps in notebook 09.

HLS scenes are expected as one folder per acquisition, named `L<YYYY-MM-DD>` or `S<YYYY-MM-DD>`,
holding the band GeoTIFFs and the Fmask layer.

In [ ]:
import re
from datetime import timedelta
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from rasterio.mask import mask

In [ ]:
HLS_DIR = Path("../data/raw/hls")                    # contains Landsat/ and Sentinel/
EXTENT_SHP = Path("../data/raw/shapefiles/study_extent/ET_Extent_Akron.shp")
VI_DIR = Path("../data/processed/vi_rasters")        # per-scene rasters
DAILY_DIR = Path("../data/processed/vi_rasters_daily")

# NIR and SWIR1 have different band numbers in the two HLS products
BANDS = {
    "Landsat":  {"prefix": "L", "blue": "B02", "red": "B04", "nir": "B05", "swir1": "B06"},
    "Sentinel": {"prefix": "S", "blue": "B02", "red": "B04", "nir": "B8A", "swir1": "B11"},
}
INDICES = ["NDVI", "EVI"]

## Cloud mask

HLS Fmask is a bit-packed QA layer. Bit 1 is cloud, bit 3 is cloud shadow and bit 4 is snow/ice.
A pixel is kept only when all three bits are 0. The function returns the Fmask values that pass,
so the check is done once per unique value instead of once per pixel.

In [ ]:
def clear_sky_values(fmask):
    good = []
    for v in np.unique(fmask):
        v = int(v)
        cloud, shadow, snow = (v >> 1) & 1, (v >> 3) & 1, (v >> 4) & 1
        if not (cloud or shadow or snow):
            good.append(v)
    return good


def find_band(folder, key):
    matches = sorted(f for f in folder.iterdir() if key in f.name)
    if not matches:
        raise FileNotFoundError(f"{key} not found in {folder.name}")
    return matches[0]

## Per-scene NDVI and EVI

Reflectance is stored as integers scaled by 10000.

In [ ]:
def ndvi_evi(blue, red, nir, fmask):
    good = clear_sky_values(fmask)
    cloudy = np.isin(fmask, good, invert=True)

    blue, red, nir = (b * 0.0001 for b in (blue, red, nir))
    for b in (blue, red, nir):
        b[cloudy] = np.nan

    with np.errstate(divide="ignore", invalid="ignore"):
        ndvi = (nir - red) / (nir + red)
        evi = 2.5 * (nir - red) / (nir + 6 * red - 7.5 * blue + 1)
    return {"NDVI": ndvi, "EVI": evi}


for sensor, bands in BANDS.items():
    for idx in INDICES:
        (VI_DIR / idx / sensor).mkdir(parents=True, exist_ok=True)

    for folder in sorted((HLS_DIR / sensor).iterdir()):
        if not folder.is_dir():
            continue
        try:
            paths = {k: find_band(folder, bands[k]) for k in ("blue", "red", "nir")}
            paths["fmask"] = find_band(folder, "Fmask")
        except FileNotFoundError as e:
            print(f"skipping {folder.name}: {e}")
            continue

        with rasterio.open(paths["red"]) as ref:
            meta = ref.meta.copy()
        arrays = {}
        for k, p in paths.items():
            with rasterio.open(p) as src:
                # Fmask stays integer so the bit checks work
                arrays[k] = src.read(1).astype("int32" if k == "fmask" else "float32")

        out = ndvi_evi(arrays["blue"], arrays["red"], arrays["nir"], arrays["fmask"])

        meta.update(dtype="float32", count=1, compress="lzw", nodata=np.nan)
        date_str = folder.name.replace(bands["prefix"], "")
        for idx, data in out.items():
            with rasterio.open(VI_DIR / idx / sensor / f"{date_str}.tif", "w", **meta) as dst:
                dst.write(data.astype("float32"), 1)

    print(f"{sensor}: done")

## Daily rasters

Landsat and Sentinel scenes are pooled into one time series per index, clipped to the study
extent, and each gap between two scenes is filled by linear interpolation, pixel by pixel.
When both sensors have a scene on the same day, the Landsat scene is used.

In [ ]:
extent = gpd.read_file(EXTENT_SHP)


def date_from_name(path):
    m = re.search(r"(\d{4}-\d{2}-\d{2})", path.name)
    return pd.Timestamp(m.group(1)) if m else None


def clip_to_extent(path):
    with rasterio.open(path) as src:
        geoms = [f["geometry"] for f in extent.to_crs(src.crs).__geo_interface__["features"]]
        arr, transform = mask(src, geoms, crop=True)
        profile = src.profile
        profile.update(height=arr.shape[1], width=arr.shape[2], transform=transform)
    return arr, profile


for idx in INDICES:
    scenes = {}
    for sensor in ["Sentinel", "Landsat"]:          # Landsat second, so it wins on shared dates
        for p in (VI_DIR / idx / sensor).glob("*.tif"):
            d = date_from_name(p)
            if d is not None:
                scenes[d] = p
    scenes = sorted(scenes.items())

    out_dir = DAILY_DIR / idx
    out_dir.mkdir(parents=True, exist_ok=True)

    for (d1, f1), (d2, f2) in zip(scenes[:-1], scenes[1:]):
        arr1, profile = clip_to_extent(f1)
        arr2, _ = clip_to_extent(f2)
        gap = (d2 - d1).days
        for k in range(1, gap):
            w = k / gap
            interp = arr1 + (arr2 - arr1) * w
            day = d1 + timedelta(days=k)
            with rasterio.open(out_dir / f"{day:%Y-%m-%d}.tif", "w", **profile) as dst:
                dst.write(interp.astype(profile["dtype"]))

    # the scene dates themselves, clipped
    for d, f in scenes:
        out_path = out_dir / f"{d:%Y-%m-%d}.tif"
        if not out_path.exists():
            arr, profile = clip_to_extent(f)
            with rasterio.open(out_path, "w", **profile) as dst:
                dst.write(arr)

    print(f"{idx}: {len(list(out_dir.glob('*.tif')))} daily rasters")